# Step 9: Analysis charts

This notebook turns Step 8's scored and ranked companies into a set of
research-report-style charts for the sector note. It only reads from
`data/processed/` and `data/companies.csv` - it never recomputes or edits a
score.

The shared visual style (colors, fonts, title/source helpers) lives in
`src/chart_style.py`, so every chart here looks like part of the same
report instead of being styled one-off.

For each chart, the markdown cell just above it explains what is plotted,
how to read it, and where any number came from. The charts' headlines state
a specific, data-checked finding (not a neutral question) - every number in
every headline was checked against `final_scores.csv` before being written
into this notebook.


In [1]:
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import StrMethodFormatter
from pathlib import Path

# Project folders, same pattern as notebooks/01_explore.ipynb.
PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
COMPANIES_FILE = PROJECT_ROOT / "data" / "companies.csv"
CHARTS_DIR = PROJECT_ROOT / "outputs" / "charts"

# Create the charts folder if it doesn't exist yet. This is the only
# filesystem change this notebook makes.
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Make the project root importable so `src` can be found, same pattern as
# tests/test_scores.py.
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from src.chart_style import (
    apply_house_style, add_title, add_source, add_callout, save_chart,
    RISK_BAND_COLORS, ALTMAN_ZONE_COLORS, DEBT_LINE_COLORS,
    STATUS_GOOD, STATUS_CAUTION, STATUS_BAD, NEUTRAL_COLOR,
    TEXT_DARK, TEXT_GREY, GRID_COLOR,
)

apply_house_style()


## Load the processed data

- `final_scores.csv`: one row per company for FY2026, with the Altman Z'',
  Piotroski F and Beneish M results already classified into zones/bands, the
  five custom red flags, and the combined risk_score / risk_band (all
  computed in Step 8 by `src/scores.py` and `src/flags.py`).
- `scores.csv`: the same scores but for every fiscal year scored (FY2025 and
  FY2026), with the underlying Piotroski test-by-test results and Beneish
  index components. It's loaded here for completeness, but none of the
  charts below need more detail than `final_scores.csv` already has for
  FY2026, so it isn't plotted.
- `ratios.csv`: financial ratios (e.g. receivable days, cash conversion,
  interest coverage) for every company and fiscal year FY2022-FY2026 - used
  for the Ajanta case study and the Piramal interest-cover callout.
- `financials_clean.csv`: the underlying cleaned financial statement line
  items (e.g. total_debt) for every company and fiscal year - used for the
  debt case study.
- `companies.csv`: ticker, full company name, short name (for chart labels)
  and sector.


In [2]:
final_scores = pd.read_csv(PROCESSED_DIR / "final_scores.csv")
scores = pd.read_csv(PROCESSED_DIR / "scores.csv")
ratios = pd.read_csv(PROCESSED_DIR / "ratios.csv")
financials = pd.read_csv(PROCESSED_DIR / "financials_clean.csv")
companies = pd.read_csv(COMPANIES_FILE)

final_scores.head()


,ticker,company_name,altman_z,altman_zone,piotroski_f,piotroski_band,beneish_m,beneish_zone,f1_receivables_outpacing_sales,f2_weak_cash_conversion,f3_inventory_buildup,f4_debt_stress,f5_low_interest_cover,risk_score,risk_band,reasons,score_notes,analyst_note
0,PPLPHARMA.NS,Piramal Pharma Ltd.,1.949170,Grey,3,Weak,-3.141829,Unlikely,0,0,1,1,1,5,High,Altman Grey (+1); Piotroski F 3 (+1); inventor...,NaN,NaN
1,ZYDUSLIFE.NS,Zydus Lifesciences Ltd.,4.440156,Safe,4,Average,-2.057144,Watch,1,1,1,1,0,5,High,Beneish Watch (+1); receivables grew 33% vs sa...,NaN,FY2026 acquisitions: 85.6% of Amplitude Surgic...
2,TORNTPHARM.NS,Torrent Pharmaceuticals Ltd.,1.393992,Grey,6,Average,-1.784497,Watch,1,0,0,1,0,4,High,Altman Grey (+1); Beneish Watch (+1); receivab...,NaN,"FY2026 debt-funded acquisition (borrowings 3,2..."
3,BIOCON.NS,Biocon Ltd.,2.508398,Grey,7,Average,-2.469228,Unlikely,0,0,1,0,1,3,Watch,Altman Grey (+1); inventory days up 44 vs sect...,NaN,NaN
4,ALKEM.NS,Alkem Laboratories Ltd.,7.341745,Safe,4,Average,-2.190214,Watch,0,1,0,1,0,3,Watch,Beneish Watch (+1); operating cash flow below ...,NaN,NaN


In [3]:
# Map each ticker to its short display name (e.g. "AJANTPHARM.NS" -> "Ajanta").
# short_name comes from data/companies.csv, which src/prepare_companies.py
# builds from a hand-picked SHORT_NAMES dict - not invented here, just read in.
SHORT_NAMES = dict(zip(companies["ticker"], companies["short_name"]))

# Every company we score should have a short name. If this fails,
# companies.csv is out of sync with final_scores.csv.
missing_names = set(final_scores["ticker"]) - set(SHORT_NAMES.keys())
assert not missing_names, f"No short name for: {missing_names}"


## Chart 0: sector summary

A one-page summary: headline counts on the left, the same risk ranking as
Chart 1 on the right. Every number in the tiles is computed from
`final_scores` in the cell below, not typed in by hand - companies screened,
how many fall into each risk_band, how many Beneish M-scores are in the
Likely (highest-concern) zone, and the sector's median Altman Z''.


In [4]:
n_companies = len(final_scores)
n_high = int((final_scores["risk_band"] == "High").sum())
n_watch = int((final_scores["risk_band"] == "Watch").sum())
n_low = int((final_scores["risk_band"] == "Low").sum())
n_beneish_likely = int((final_scores["beneish_zone"] == "Likely").sum())
median_altman_z = final_scores["altman_z"].median()

# (number, caption, status chip color or None)
tiles = [
    (str(n_companies), "Companies screened", None),
    (str(n_high), "High risk", RISK_BAND_COLORS["High"]),
    (str(n_watch), "Watch", RISK_BAND_COLORS["Watch"]),
    (str(n_low), "Low risk", RISK_BAND_COLORS["Low"]),
    (str(n_beneish_likely), 'Beneish "Likely"', STATUS_BAD),
    (f"{median_altman_z:.1f}", "Median Altman Z''", None),
]

fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(
    1, 2, width_ratios=[1, 1.3], left=0.03, right=0.98, top=0.76, bottom=0.10, wspace=0.1
)

# --- Left: headline-number tiles ---
ax_tiles = fig.add_subplot(gs[0, 0])
ax_tiles.set_xlim(0, 1)
ax_tiles.set_ylim(0, 1)
ax_tiles.axis("off")

# Rows pulled up near the top of the panel (rather than vertically centered)
# so the tiles sit level with the top of the bar chart next to them.
tile_centers = [(0.17, 0.80), (0.5, 0.80), (0.83, 0.80), (0.17, 0.34), (0.5, 0.34), (0.83, 0.34)]
for (x, y), (number, caption, chip_color) in zip(tile_centers, tiles):
    if chip_color is not None:
        ax_tiles.add_patch(Rectangle((x - 0.07, y + 0.13), 0.14, 0.018, color=chip_color))
    ax_tiles.text(x, y, number, fontsize=30, fontweight="bold", color=TEXT_DARK, ha="center", va="center")
    ax_tiles.text(x, y - 0.14, caption, fontsize=10, color=TEXT_GREY, ha="center", va="center")

# --- Right: risk ranking bar chart (same data and sort as Chart 1) ---
ax_bars = fig.add_subplot(gs[0, 1])
chart0_df = final_scores.sort_values(
    ["risk_score", "altman_z"], ascending=[False, True]
).reset_index(drop=True)

y_pos = range(len(chart0_df))
bar_colors = [RISK_BAND_COLORS[band] for band in chart0_df["risk_band"]]
bars = ax_bars.barh(y_pos, chart0_df["risk_score"], color=bar_colors)
ax_bars.bar_label(bars, padding=3, fontsize=9)

ax_bars.set_yticks(list(y_pos))
ax_bars.set_yticklabels([SHORT_NAMES[t] for t in chart0_df["ticker"]], fontsize=9)
ax_bars.invert_yaxis()
ax_bars.set_xlim(0, chart0_df["risk_score"].max() + 1)
ax_bars.set_xlabel("Risk score (points)")
ax_bars.set_title("Risk score by company", fontsize=12, color=TEXT_DARK, loc="left")
ax_bars.grid(axis="x", color=GRID_COLOR, linewidth=0.8)

legend_handles = [Patch(color=RISK_BAND_COLORS[band], label=band) for band in ["Low", "Watch", "High"]]
ax_bars.legend(handles=legend_handles, title="Risk band", loc="lower right")

add_title(
    fig,
    "Nifty Pharma red-flag screen: 3 of 20 companies screen High risk",
    "Combined Altman, Piotroski, Beneish and custom red-flag score, FY2026",
)
add_source(fig)

save_chart(fig, CHARTS_DIR / "00_sector_summary.png")


## Chart 1: risk ranking

`risk_score` (from Step 8) adds one point for each signal: Altman Z'' in the
Distress zone (+2) or Grey zone (+1), Beneish M in the Likely zone (+2) or
Watch zone (+1), a Piotroski F-score of 3 or below (+1), and each of the
five custom red flags F1-F5 (+1 each). Bars are sorted with the highest
score at the top; color shows the risk_band each score falls into.


In [5]:
chart1_df = final_scores.sort_values(
    ["risk_score", "altman_z"], ascending=[False, True]
).reset_index(drop=True)

fig, ax = plt.subplots()
fig.subplots_adjust(top=0.80, bottom=0.10, left=0.15, right=0.95)

y_pos = range(len(chart1_df))
bar_colors = [RISK_BAND_COLORS[band] for band in chart1_df["risk_band"]]
bars = ax.barh(y_pos, chart1_df["risk_score"], color=bar_colors)
ax.bar_label(bars, padding=3)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([SHORT_NAMES[t] for t in chart1_df["ticker"]])
# sort_values() put the highest score first (top of the list);
# invert_yaxis() keeps it at the top of the chart too.
ax.invert_yaxis()

# A little headroom on the right so the value labels aren't clipped.
ax.set_xlim(0, chart1_df["risk_score"].max() + 1)

ax.set_xlabel("Risk score (points, higher = more red flags triggered)")
ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)

legend_handles = [
    Patch(color=RISK_BAND_COLORS[band], label=band) for band in ["Low", "Watch", "High"]
]
ax.legend(handles=legend_handles, title="Risk band")

add_title(
    fig,
    "Three companies screen High risk, two of them because of acquisitions",
    "Combined risk score, FY2026 (higher = more red flags)",
)
add_source(fig)

save_chart(fig, CHARTS_DIR / "01_risk_ranking.png")


## Chart 2: Altman Z''

Altman Z'' is a solvency score built from working capital, retained
earnings, EBIT and sales, each scaled by total assets (the "double-prime"
version is designed for private/service-type and non-manufacturing-heavy
firms, so it doesn't use market value of equity). A **lower** Z'' means a
company's balance sheet looks more like firms that have historically gone
into financial distress. Bar color shows the zone each score falls into
(same thresholds as Step 8: below 1.1 = Distress, 1.1-2.6 = Grey, above
2.6 = Safe). Bars are sorted with the lowest (most concerning) Z'' at the
top.


In [6]:
chart2_df = final_scores.sort_values("altman_z", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots()
fig.subplots_adjust(top=0.74, bottom=0.10, left=0.15, right=0.95)

y_pos = range(len(chart2_df))
bar_colors = [ALTMAN_ZONE_COLORS[zone] for zone in chart2_df["altman_zone"]]
bars = ax.barh(y_pos, chart2_df["altman_z"], color=bar_colors)
ax.bar_label(bars, fmt="%.1f", padding=3)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([SHORT_NAMES[t] for t in chart2_df["ticker"]])
ax.invert_yaxis()  # lowest (most concerning) Z'' ends up at the top

ax.axvline(1.1, color="grey", linestyle="--", linewidth=1)
ax.axvline(2.6, color="grey", linestyle="--", linewidth=1)

# Place the zone labels just above the plot area (axes-fraction y, data x) at
# the midpoint of each zone, so they land in the right place no matter how
# wide the x-axis ends up being.
x_min, x_max = ax.get_xlim()
trans = ax.get_xaxis_transform()
ax.text((x_min + 1.1) / 2, 1.02, "Distress", transform=trans, ha="center", va="bottom", fontsize=9, color=TEXT_GREY)
ax.text((1.1 + 2.6) / 2, 1.02, "Grey", transform=trans, ha="center", va="bottom", fontsize=9, color=TEXT_GREY)
ax.text((2.6 + x_max) / 2, 1.02, "Safe", transform=trans, ha="center", va="bottom", fontsize=9, color=TEXT_GREY)

# A little headroom on the right so the value labels aren't clipped.
ax.set_xlim(x_min, x_max + 1)

ax.set_xlabel("Altman Z'' score (unitless)")
ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)

legend_handles = [
    Patch(color=ALTMAN_ZONE_COLORS[zone], label=zone) for zone in ["Distress", "Grey", "Safe"]
]
# Bars are sorted shortest-at-top, so the top-right corner is the one part
# of the plot no bar's value label reaches into.
ax.legend(handles=legend_handles, title="Altman zone", loc="upper right")

add_title(
    fig,
    "17 of 20 companies are financially Safe; Torrent, Piramal and Biocon sit in the Grey zone",
    "Altman Z'' score, FY2026. Below 1.1 = Distress, 1.1-2.6 = Grey, above 2.6 = Safe",
)
add_source(fig)

save_chart(fig, CHARTS_DIR / "02_altman.png")


## Chart 3: Beneish M-score

The Beneish M-score combines eight accrual- and ratio-based indices (e.g.
days sales in receivables, gross margin, asset quality, sales growth) into a
single number that flags accounting patterns associated with past known
earnings-manipulation cases. A **higher** (less negative) M-score is more
concerning. This is a dot plot rather than bars, because a bar starting at 0
would make the least suspicious company look the "biggest" - only the dot's
position matters. Dots are sorted with the highest (most concerning)
M-score at the top; "Unlikely" dots are neutral grey here (rather than
green) so the Watch/Likely dots - the ones actually worth a second look -
stand out. The three callouts quote this project's own analyst notes
(`data/analyst_notes.csv`), not a new judgement made in this chart.


In [7]:
chart3_df = final_scores.sort_values("beneish_m", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots()
fig.subplots_adjust(top=0.74, bottom=0.10, left=0.15, right=0.95)

y_pos = list(range(len(chart3_df)))

ax.axvline(-1.78, color="grey", linestyle="--", linewidth=1)
ax.axvspan(-2.22, -1.78, color=STATUS_CAUTION, alpha=0.12)

dot_colors_map = {"Unlikely": NEUTRAL_COLOR, "Watch": STATUS_CAUTION, "Likely": STATUS_BAD}
dot_colors = [dot_colors_map[zone] for zone in chart3_df["beneish_zone"]]
ax.scatter(chart3_df["beneish_m"], y_pos, color=dot_colors, s=70, zorder=3)

for y, m_score in zip(y_pos, chart3_df["beneish_m"]):
    ax.annotate(
        f"{m_score:.2f}", (m_score, y),
        textcoords="offset points", xytext=(8, 0), va="center", fontsize=9,
    )

ax.set_yticks(y_pos)
ax.set_yticklabels([SHORT_NAMES[t] for t in chart3_df["ticker"]])
# Extra headroom above row 0 (Abbott) for its callout, which points upward.
ax.set_ylim(len(chart3_df) - 0.5, -1.3)

ax.set_xlim(-3.3, -1.4)

trans = ax.get_xaxis_transform()
ax.text((-2.22 + -1.78) / 2, 1.02, "Watch", transform=trans, ha="center", va="bottom", fontsize=9, color=TEXT_GREY)

ax.set_xlabel("Beneish M-score (unitless)")
ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)

legend_handles = [
    Patch(color=dot_colors_map[zone], label=zone) for zone in ["Unlikely", "Watch", "Likely"]
]
ax.legend(handles=legend_handles, title="Beneish zone", loc="lower right")

# Callouts for the three dots the headline is about. xy is each company's
# own (M-score, row position); xytext is a screen-point offset from there.
callouts = [
    ("ABBOTINDIA.NS", "Likely false positive:\ndeposits reclassified", (50, 30)),
    # A purely horizontal offset would send the leader line straight through
    # Torrent's own value label (right next to its dot), so this one needs
    # a vertical component too, the same as the other two.
    ("TORNTPHARM.NS", "Acquisition effects", (80, 18)),
    ("AJANTPHARM.NS", "Working-capital\ndeterioration", (60, -24)),
]
for ticker, text, xytext in callouts:
    row_position = chart3_df.index[chart3_df["ticker"] == ticker][0]
    m_score = chart3_df.loc[row_position, "beneish_m"]
    add_callout(ax, (m_score, row_position), text, xytext)

add_title(
    fig,
    "Only Abbott crosses the manipulation threshold, and the driver looks benign",
    "Beneish M-score, FY2026. Above -1.78 = Likely, -2.22 to -1.78 = Watch",
)
add_source(fig)

save_chart(fig, CHARTS_DIR / "03_beneish.png")


## Chart 4: Altman Z'' vs Piotroski F-score

This chart puts two independent signals from Step 8 on one plot: Altman Z''
(x-axis, a point-in-time solvency score) against Piotroski F-score (y-axis, a
0-9 count of fundamental tests passed - e.g. positive ROA, improving margins,
no new share issuance - where a higher score means more tests point to an
improving business). The dashed lines mark the same Altman Z'' zones as
Chart 2 (labelled inside the plot, near the top - the Distress label is
rotated because that zone is narrow) and the same Piotroski bands as the
piotroski_band column (labelled along the right at F = 3, 5.5 and 8.5: Weak
<= 3, Average 4-7, Strong >= 8). Dot color shows risk_band, the same colors
as Chart 1. Piramal's interest coverage is pulled from `ratios.csv` for the
callout, not typed in by hand.


In [8]:
piramal_interest_cover = ratios.loc[
    (ratios["ticker"] == "PPLPHARMA.NS") & (ratios["fiscal_year"] == 2026), "interest_coverage"
].iloc[0]

fig, ax = plt.subplots(figsize=(10, 9))
fig.subplots_adjust(top=0.80, bottom=0.08, left=0.10, right=0.84)

ax.axvline(1.1, color="grey", linestyle="--", linewidth=1)
ax.axvline(2.6, color="grey", linestyle="--", linewidth=1)
ax.axhline(3.5, color="grey", linestyle="--", linewidth=1)
ax.axhline(7.5, color="grey", linestyle="--", linewidth=1)

dot_colors = [RISK_BAND_COLORS[band] for band in final_scores["risk_band"]]
ax.scatter(
    final_scores["altman_z"], final_scores["piotroski_f"],
    color=dot_colors, s=60, zorder=3,
)

# Capture the x limits now that the real data has been plotted (not right
# after the threshold lines, which would autoscale to just the lines
# themselves and could clip off a real point - this clipped Piramal, the
# company at F=3, on an earlier version of this chart).
x_min, x_max = ax.get_xlim()

# Fixed range: comfortably covers every point (F runs 3-9 here) with a
# little headroom below the lowest point and - more importantly - above the
# highest, for the Altman zone labels, so they sit inside the plot and
# never collide with the chart title.
ax.set_ylim(2.5, 9.5)
TOP_LABEL_LOW = 9.15
TOP_LABEL_HIGH = 9.35

# The Distress zone (xmin to 1.1) is too narrow for "Distress" to fit
# horizontally, so it's rotated; Grey and Safe are staggered to a different
# height so neither touches the rotated label next to it. Safe's own
# midpoint x happens to land almost exactly on IPCA's dot, so it uses the
# same (higher) row as Grey rather than Distress's row - IPCA's own label
# sits just above its dot, closer to the lower row.
ax.text((x_min + 1.1) / 2, TOP_LABEL_LOW, "Distress", rotation=90, ha="center", va="center", fontsize=9, color=TEXT_GREY)
ax.text((1.1 + 2.6) / 2, TOP_LABEL_HIGH, "Grey", ha="center", va="center", fontsize=9, color=TEXT_GREY)
ax.text((2.6 + x_max) / 2, TOP_LABEL_HIGH, "Safe", ha="center", va="center", fontsize=9, color=TEXT_GREY)

# Piotroski band labels along the right (axes-fraction x, data y), centered
# in each band rather than sitting on the threshold lines.
trans_y = ax.get_yaxis_transform()
ax.text(1.02, 3, "Weak", transform=trans_y, ha="left", va="center", fontsize=9, color=TEXT_GREY)
ax.text(1.02, 5.5, "Average", transform=trans_y, ha="left", va="center", fontsize=9, color=TEXT_GREY)
ax.text(1.02, 8.5, "Strong", transform=trans_y, ha="left", va="center", fontsize=9, color=TEXT_GREY)

ax.grid(color=GRID_COLOR, linewidth=0.8, zorder=0)

# A handful of points sit close enough together (in Altman Z'' / Piotroski F
# space) that the default label offset would overlap a neighbor, or - for
# Divi's, the rightmost point - would run past the edge of the plot. Nudge
# just those by hand and draw a thin leader line so it's still clear which
# label belongs to which dot; everyone else gets the same small default
# offset.
DEFAULT_OFFSET = (6, 6)
LABEL_OFFSET_OVERRIDES = {
    "SUNPHARMA.NS": (-45, 14),
    "AJANTPHARM.NS": (12, 16),
    "GLAND.NS": (-45, 10),
    # Piramal sits at the bottom of the plot (F=3, the lowest score), so its
    # two-line label needs to go up, not down (down would run into the
    # x-axis tick labels) - and up far enough to clear both the F=3.5 "Weak"
    # threshold line and Zydus's own label/dot just to its right.
    "PPLPHARMA.NS": (22, 95),
    "DIVISLAB.NS": (-45, 10),
}

for _, row in final_scores.iterrows():
    ticker = row["ticker"]
    label = SHORT_NAMES[ticker]
    if ticker == "PPLPHARMA.NS":
        # Piramal is the headline's subject - call out its interest cover
        # alongside its name, not just the name.
        label = f"{label}\nInterest cover {piramal_interest_cover:.2f}x"
    offset = LABEL_OFFSET_OVERRIDES.get(ticker, DEFAULT_OFFSET)
    arrowprops = None
    if ticker in LABEL_OFFSET_OVERRIDES:
        arrowprops = dict(arrowstyle="-", color="#999999", linewidth=0.75, shrinkA=0, shrinkB=4)
    ax.annotate(
        label,
        (row["altman_z"], row["piotroski_f"]),
        textcoords="offset points",
        xytext=offset,
        fontsize=9,
        zorder=4,
        arrowprops=arrowprops,
    )

ax.set_xlabel("Altman Z'' score (unitless)")
ax.set_ylabel("Piotroski F-score (0-9 tests passed)")

legend_handles = [
    Patch(color=RISK_BAND_COLORS[band], label=band) for band in ["Low", "Watch", "High"]
]
ax.legend(handles=legend_handles, title="Risk band", loc="upper left", bbox_to_anchor=(1.12, 1))

add_title(
    fig,
    "Piramal is the only company that is both financially weak and weakening",
    "Financial health (Altman Z'') vs momentum (Piotroski F), FY2026",
)
add_source(fig)

save_chart(fig, CHARTS_DIR / "04_altman_vs_piotroski.png")


## Chart 5: red-flag heatmap

Every signal that adds to `risk_score` is shown here as its own 0/1 flag:
whether Altman Z'' is Grey or Distress (not Safe), whether the Piotroski band
is Weak, whether the Beneish zone is Watch or Likely, and the five custom
flags from Step 8 (F1 receivables outpacing sales, F2 weak cash conversion,
F3 inventory buildup, F4 debt stress, F5 low interest cover). Rows are
companies, sorted the same way as Chart 1 (highest risk_score first). A dark
cell means that flag is triggered (1), a light cell means it isn't (0), and
grey means the underlying data was missing (NaN) - this project never
invents a value for a missing flag. The **Total flags** column on the right
is the row sum of the 8 flag columns, computed in the cell below.


In [9]:
FLAG_COLOR = STATUS_BAD
NO_FLAG_COLOR = "#f2f1ee"
NA_COLOR = NEUTRAL_COLOR

heat_df = final_scores.copy()

# Three of the eight flag columns don't exist yet as plain 0/1 columns -
# derive them here from the zone/band columns Step 8 already computed. A
# company flagged "Grey" is not Safe, so it still counts as a concern for
# this heatmap even though it isn't full Distress.
heat_df["altman_flag"] = (heat_df["altman_zone"] != "Safe").astype(float)
heat_df["piotroski_flag"] = (heat_df["piotroski_band"] == "Weak").astype(float)
heat_df["beneish_flag"] = heat_df["beneish_zone"].isin(["Watch", "Likely"]).astype(float)

flag_columns = [
    "altman_flag", "piotroski_flag", "beneish_flag",
    "f1_receivables_outpacing_sales", "f2_weak_cash_conversion",
    "f3_inventory_buildup", "f4_debt_stress", "f5_low_interest_cover",
]
column_labels = [
    "Altman Grey/Distress", "Piotroski Weak", "Beneish Watch/Likely",
    "F1 Receivables", "F2 Cash conversion", "F3 Inventory",
    "F4 Debt stress", "F5 Interest cover",
]
n_cols = len(flag_columns)

heat_df = heat_df.sort_values(
    ["risk_score", "altman_z"], ascending=[False, True]
).reset_index(drop=True)
flag_matrix = heat_df[flag_columns].astype(float).to_numpy()
total_flags = np.nansum(flag_matrix, axis=1)

# Mask any NaN so it gets its own color instead of being plotted as 0 or 1.
masked_matrix = np.ma.masked_invalid(flag_matrix)
cmap = plt.matplotlib.colors.ListedColormap([NO_FLAG_COLOR, FLAG_COLOR])
cmap.set_bad(color=NA_COLOR)

fig, ax = plt.subplots(figsize=(12, 8.5))
fig.subplots_adjust(top=0.78, bottom=0.22, left=0.14, right=0.85)

ax.imshow(masked_matrix, cmap=cmap, vmin=0, vmax=1, aspect="auto")

# Label any missing cell directly, so "missing" is visible even in grayscale.
for row_index, col_index in np.argwhere(np.isnan(flag_matrix)):
    ax.text(col_index, row_index, "NA", ha="center", va="center", fontsize=8)

# Total flags column: a thin separator line, then the row sum as plain text
# (not part of the colored flag grid).
ax.axvline(n_cols - 0.5, color="#444444", linewidth=1)
for row_index, total in enumerate(total_flags):
    ax.text(n_cols, row_index, f"{int(total)}", ha="center", va="center", fontsize=10, fontweight="bold", color=TEXT_DARK)

ax.set_yticks(range(len(heat_df)))
ax.set_yticklabels([SHORT_NAMES[t] for t in heat_df["ticker"]])
ax.set_xticks(list(range(n_cols)) + [n_cols])
ax.set_xticklabels(column_labels + ["Total flags"], rotation=45, ha="right")
ax.set_xlim(-0.5, n_cols + 0.7)

# Thin white gridlines between cells, drawn on the minor ticks so they don't
# collide with the major ticks used for the labels.
ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(heat_df), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1)
ax.tick_params(which="minor", length=0)

legend_handles = [
    Patch(color=FLAG_COLOR, label="Flagged"),
    Patch(color=NO_FLAG_COLOR, label="Not flagged"),
    Patch(color=NA_COLOR, label="Not available"),
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.15, 1))

add_title(
    fig,
    "Six companies trigger no red flags; the top three carry four or five each",
    "Red flags triggered per company, FY2026",
)
add_source(fig)

save_chart(fig, CHARTS_DIR / "05_flag_heatmap.png")


## Chart 6: two case studies

**Left column - Ajanta Pharma.** Top: receivable days, how many days, on
average, Ajanta takes to collect cash from customers after a sale. Bottom:
cash conversion (operating cash flow divided by net income); the dashed line
at 1.0 marks the point where profit is fully backed by cash. The two
measures use very different scales (tens of days vs. a ratio around 1), so
they're two separate panels rather than one chart with two y-axes.

**Right - total borrowings at Torrent, Zydus and Piramal, FY2023-FY2026.**
`total_debt` (in Rs crore) is plotted as one line per company, labelled
directly at the end of each line instead of in a legend. The two callouts
quote this project's own analyst notes (`data/analyst_notes.csv`).


In [10]:
# subplot_mosaic lays out three panels: "receivable" and "cash" stacked in
# the left column, "debt" spanning the full height on the right. constrained
# layout keeps titles/labels from the three panels clear of each other.
fig, axd = plt.subplot_mosaic(
    [["receivable", "debt"], ["cash", "debt"]],
    figsize=(14, 7.5), constrained_layout=True,
)
# The top of this rect leaves room for the (two-line) headline and
# subtitle; it's lower than the other charts' so there's a visible gap
# between the subtitle and the "Ajanta: receivable days" panel title below it.
fig.get_layout_engine().set(rect=(0.0, 0.06, 1.0, 0.72))
ax_receivable = axd["receivable"]
ax_cash = axd["cash"]
ax_right = axd["debt"]

# --- Left column, top: Ajanta's receivable days ---
ajanta = ratios[
    (ratios["ticker"] == "AJANTPHARM.NS") & (ratios["fiscal_year"].between(2023, 2026))
].sort_values("fiscal_year")

ax_receivable.plot(ajanta["fiscal_year"], ajanta["receivable_days"], marker="o", color=DEBT_LINE_COLORS["TORNTPHARM.NS"])
ax_receivable.set_xticks(ajanta["fiscal_year"])
ax_receivable.set_ylabel("Receivable days (days)")
ax_receivable.set_title("Ajanta: receivable days", fontsize=12, color=TEXT_DARK, loc="left")
ax_receivable.grid(axis="y", color=GRID_COLOR, linewidth=0.8)

# --- Left column, bottom: Ajanta's cash conversion ---
ax_cash.plot(ajanta["fiscal_year"], ajanta["cash_conversion"], marker="s", color=DEBT_LINE_COLORS["ZYDUSLIFE.NS"])
ax_cash.axhline(1.0, color="grey", linestyle="--", linewidth=1)
# Placed at the first fiscal year, where the line sits well above 1.0, so
# the label doesn't sit on top of a data point.
ax_cash.text(
    ajanta["fiscal_year"].iloc[0], 1.0, " profit fully backed by cash",
    ha="left", va="bottom", fontsize=9, color=TEXT_GREY,
)
ax_cash.set_xticks(ajanta["fiscal_year"])
ax_cash.set_xlabel("Fiscal year")
ax_cash.set_ylabel("Cash conversion (ratio)")
ax_cash.set_title("Ajanta: cash conversion (CFO / net income)", fontsize=12, color=TEXT_DARK, loc="left")
ax_cash.grid(axis="y", color=GRID_COLOR, linewidth=0.8)

# --- Right: total debt at Torrent, Zydus and Piramal, direct end labels ---
case_tickers = ["TORNTPHARM.NS", "ZYDUSLIFE.NS", "PPLPHARMA.NS"]
debt = financials[
    (financials["ticker"].isin(case_tickers)) & (financials["fiscal_year"].between(2023, 2026))
]

for ticker in case_tickers:
    company_debt = debt[debt["ticker"] == ticker].sort_values("fiscal_year")
    ax_right.plot(
        company_debt["fiscal_year"], company_debt["total_debt"],
        marker="o", color=DEBT_LINE_COLORS[ticker],
    )
    last_row = company_debt.iloc[-1]
    # Direct label at the line's end instead of a legend. Text is always
    # dark, not the line's own color.
    ax_right.annotate(
        SHORT_NAMES[ticker],
        (last_row["fiscal_year"], last_row["total_debt"]),
        textcoords="offset points", xytext=(8, 0), va="center",
        fontsize=10, fontweight="bold", color=TEXT_DARK,
    )

ax_right.set_xlabel("Fiscal year")
ax_right.set_ylabel("Total debt (Rs crore)")
ax_right.set_xticks(sorted(debt["fiscal_year"].unique()))
ax_right.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax_right.grid(axis="y", color=GRID_COLOR, linewidth=0.8)

x_min, x_max = ax_right.get_xlim()
ax_right.set_xlim(x_min, x_max + 0.9)  # headroom for the end labels and callouts

torrent_last = debt[debt["ticker"] == "TORNTPHARM.NS"].sort_values("fiscal_year").iloc[-1]
zydus_last = debt[debt["ticker"] == "ZYDUSLIFE.NS"].sort_values("fiscal_year").iloc[-1]
add_callout(
    ax_right, (torrent_last["fiscal_year"], torrent_last["total_debt"]),
    "Debt-funded acquisition (FY2026)", (-120, 25),
)
add_callout(
    ax_right, (zydus_last["fiscal_year"], zydus_last["total_debt"]),
    "Amplitude Surgical +\nAgenus facilities (FY2026)", (-170, -45),
)

add_title(
    fig,
    "Ajanta's cash conversion collapsed in FY2026, while Torrent and Zydus borrowed heavily for acquisitions",
    "Left: Ajanta working capital, FY2023-26. Right: total debt (Rs crore), FY2023-26",
)
add_source(fig)

save_chart(fig, CHARTS_DIR / "06_case_studies.png")


## Done

All seven charts have been saved to `outputs/charts/`. The cell below lists
the files as a quick check that nothing is missing.


In [11]:
sorted(path.name for path in CHARTS_DIR.glob("*.png"))


['00_sector_summary.png',
 '01_risk_ranking.png',
 '02_altman.png',
 '03_beneish.png',
 '04_altman_vs_piotroski.png',
 '05_flag_heatmap.png',
 '06_case_studies.png']